# Giai đoạn 3A: Baseline FINE-TUNE cùng dữ liệu để so sánh công bằng với hướng cải tiến
Notebook này trả lời câu hỏi còn thiếu của Giai đoạn 2: *"nếu fine-tune theo cách đơn giản/tiêu chuẩn trên cùng 2.000 trang V-Doc thì được bao nhiêu?"*

| Mã | Baseline | Tham số train | Nén | Seed |
|---|---|---|---|---|
| `ZS` | ColQwen2.5-3B zero-shot, **có / không** `'Query: '` | 0 | ✗ / Hier×4 | – |
| `FT-E1` | Linear 128×128 phía query + Hier×4 | ~16,5K | Hier×4 | 3 |
| `FT-E1b` | Linear 2 phía + Hier×4 | ~33K | Hier×4 | 3 |
| `FT-E2` | Nhánh query của adapter v2 (subword + gate) + Hier×4 | ~23K | Hier×4 | 3 |
| `FT-E3` | Linear phía query, không nén | ~16,5K | ✗ | 3 |
| `FT-LoRA` | LoRA r=32 trên chính backbone 3B (loss gốc ColPali) | ~1e8 | ✗ / Hier×4 | 1 |

- Cùng dữ liệu, cùng hard negatives (khai thác bằng ColQwen2.5 + `'Query: '`), cùng lịch train (AdamW 5e-4, warmup 1 + cosine 12 epoch, chọn epoch theo val) với Giai đoạn 2.
- Đánh giá: V-Doc test (1.067, cùng miền), EN DocVQA (500), **OOD ViInfographicVQA** (1.374).
- Chạy notebook này **trước** `stage3_adapter_v3_fixed.ipynb` (notebook kia đọc kết quả ở đây để làm bảng tổng hợp cuối). Dùng chung cache trong `SAVE_DIR`; chạy lại sẽ tự bỏ qua phần đã có.

In [ ]:
!pip install -q "colpali-engine==0.3.18" scipy pytesseract rank_bm25 timm einops matplotlib
!pip uninstall -y -q torchao
!pip install -q eva-decord || pip install -q decord || true

In [ ]:
import os, gc, glob, json, time, random, hashlib, tarfile, re, subprocess, shutil, copy, traceback
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download, login
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
from transformers import get_cosine_schedule_with_warmup

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ---- Dán HuggingFace Token trực tiếp tại đây nếu chạy nhánh 5CD-AI (NHỚ XÓA TOKEN TRƯỚC KHI NỘP / PUSH GITHUB) ----
HF_TOKEN = ""

# ---- Thư mục lưu trữ dùng chung cache Giai đoạn 1 (ghi kết quả Giai đoạn 2 ra file JSON riêng) ----
_colab_secret_tok = ""
if os.path.isdir("/kaggle/working"):
    SAVE_DIR, DATA_DIR = "/kaggle/working/ViColPali_Stage1_Fresh", "/tmp/vdoc"
elif os.path.exists("/var/colab/hostname"):
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    SAVE_DIR, DATA_DIR = "/content/drive/MyDrive/ViColPali_Stage1_Fresh", "/content/vdoc"
    try:
        _colab_secret_tok = userdata.get("HF_TOKEN") or ""
    except Exception:
        _colab_secret_tok = ""
else:
    SAVE_DIR, DATA_DIR = "./ViColPali_Stage1_Fresh", "./vdoc"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

_hf_tok = (HF_TOKEN or _colab_secret_tok or os.environ.get("HF_TOKEN", "")).strip()
if _hf_tok:
    os.environ["HF_TOKEN"] = _hf_tok
    login(token=_hf_tok, add_to_git_credential=False)
    print("✓ Đã xác thực HuggingFace bằng HF_TOKEN.")
else:
    print("Chưa cung cấp HF_TOKEN (V-Doc/ColQwen tải public bình thường; nhánh 5CD-AI gated cần HF_TOKEN).")

# ---- Hằng số chia dữ liệu (giữ nguyên 100% với best.ipynb & Giai đoạn 1) ----
SEED = 42
N_SHARDS = 1
BLOCK, TEST_EVERY = 200, 10
MAX_TEST = 1500
DATA_SIZES = [1000]
N_TRAIN_PAGES, MAX_Q_PER_PAGE, N_VAL = 2000, 2, 300

# ---- Tham số huấn luyện (GIỮ NGUYÊN Giai đoạn 2 để so sánh công bằng) ----
SEEDS = [0, 1, 2]
EPOCHS = 12
WARMUP_EPOCHS = 1
LR_ADAPTER = 5e-4
WEIGHT_DECAY = 2e-2
BQ = 32
HARD_NEG = 7
TEMP_CE = 0.10
TEMP_KD = 0.15
ANCHOR_W = 0.05
SCORE_CHUNK = 8
DOC_CHUNK = 64

# ---- Cấu hình trộn dữ liệu đa miền sạch 5CD-AI ----
RUN_5CD_MIX = True
N_5CD_PAGES = 1500
MAX_PER_5CD_SOURCE = 500
MAX_5CD_SCAN = 25000

# ---- Công tắc chạy từng phần ----
RUN_COLVINTERN = False  # ColVintern-1B-v1 custom code không tương thích transformers 5.x trên Colab mới ('all_tied_weights_keys')
RUN_HIER_QWEN = True
RUN_QWEN_ADAPTERS = True
RUN_GATE_VIS = True
RUN_EN_CHECK = True
RUN_SLICE_ANALYSIS = True
RUN_PLOT = True
PREFER_FP16_ON_T4 = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0.0
VINTERN_IMG_BS = 8 if VRAM_GB >= 22 else 4
QWEN_IMG_BS = 4 if VRAM_GB >= 22 else 2

STAGE1_RES_PATH = f"{SAVE_DIR}/results_s{N_SHARDS}_b{BLOCK}_t{TEST_EVERY}_m{MAX_TEST}.json"

print("SAVE_DIR =", SAVE_DIR, "| DATA_DIR =", DATA_DIR)
if torch.cuda.is_available():
    print(
        f"GPU: {torch.cuda.get_device_name()} ({VRAM_GB:.1f} GB VRAM) | "
        f"bf16 supported: {torch.cuda.is_bf16_supported()} | "
        f"batch_size (Vintern/Qwen): {VINTERN_IMG_BS}/{QWEN_IMG_BS}"
    )
else:
    print("Đang chạy trên CPU")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
# =====================================================================================
# ---- Bổ sung Giai đoạn 3 (dùng chung cho 2 notebook stage3_*) ----
# =====================================================================================
# (1) Sửa giao thức query: colpali-engine >= 0.3.13 KHÔNG còn tự thêm tiền tố "Query: " (query_prefix = ""),
#     trong khi vidore/colqwen2.5-v0.2 được huấn luyện với "Query: " + text + 10×<|endoftext|> (xem model card).
QUERY_PREFIX = "Query: "
QWEN_ID = "vidore/colqwen2.5-v0.2"

# (2) Bộ test tiếng Việt NGOÀI MIỀN: ViInfographicVQA (single-image test: 193 ảnh / 1.374 câu hỏi người kiểm duyệt)
RUN_OOD = True
OOD_REPO = "duytranus/ViInfographicVQA"
OOD_MAX_Q = 1374

# (3) Thống kê: randomization test có cặp (Smucker et al., 2007) + hiệu chỉnh Holm; p = (b+1)/(B+1), không bao giờ in 0
N_PERM = 50000
N_BOOT = 2000

# ---- Đường dẫn cache dùng chung (tái sử dụng nguyên cache Giai đoạn 2, KHÔNG ghi đè) ----
QWEN_TEST_CACHE = f"{SAVE_DIR}/colqwen25_v02_test_emb.pt"
QWEN_TRVAL_CACHE = f"{SAVE_DIR}/colqwen25_v02_trval_s{N_SHARDS}_tr{N_TRAIN_PAGES}x{MAX_Q_PER_PAGE}_v{N_VAL}.pt"
QPREFIX_CACHE = f"{SAVE_DIR}/stage3_colqwen25_v02_queries_prefix_s{N_SHARDS}_tr{N_TRAIN_PAGES}x{MAX_Q_PER_PAGE}_v{N_VAL}.pt"
OOD_CACHE = f"{SAVE_DIR}/stage3_colqwen25_v02_ood_viinfographic_q{OOD_MAX_Q}.pt"
HIER_CACHE = f"{SAVE_DIR}/stage3_colqwen25_v02_hier4_s{N_SHARDS}_tr{N_TRAIN_PAGES}x{MAX_Q_PER_PAGE}_v{N_VAL}.pt"
RES_V2_PATH = f"{SAVE_DIR}/results_stage2_s{N_SHARDS}_b{BLOCK}_t{TEST_EVERY}_m{MAX_TEST}.json"
RES_A_PATH = f"{SAVE_DIR}/results_stage3_baselines_s{N_SHARDS}_b{BLOCK}_t{TEST_EVERY}_m{MAX_TEST}.json"
RES_B_PATH = f"{SAVE_DIR}/results_stage3_v3_s{N_SHARDS}_b{BLOCK}_t{TEST_EVERY}_m{MAX_TEST}.json"

# ---- Công tắc riêng notebook BASELINE ----
NEED_HIER_TRAIN = True          # cần Hier×4 cho tập train/val để train adapter trên trang đã nén
RUN_EMBED_BASELINES = True      # E1, E1b, E2, E3 (3 seed, nhanh)
RUN_LORA = True                 # LoRA r=32 trên backbone 3B (1 seed, ~1–2 giờ trên L4)
LORA_SEED = 0
LORA_EPOCHS = 1                 # 1 lượt qua 4.000 câu hỏi train
LORA_LR = 5e-5
LORA_EFFECTIVE_BS = 32
LORA_MICRO_BS = 4 if VRAM_GB >= 22 else 2
LORA_MAX_HOURS = 3.0
RES_PATH = RES_A_PATH
print("RES_PATH (notebook này) =", RES_PATH)

## 1. Tải `V-Doc` + `EN DocVQA` & băm MD5 (GIỮ NGUYÊN 100% Giai đoạn 2)
Cùng hằng số chia (`BLOCK = 200`, `TEST_EVERY = 10`, `SEED = 42`) → cùng 2.000 trang train / 300 val / 1.067 test / 500 EN như Giai đoạn 2, nên mọi so sánh có cặp (paired) theo từng câu hỏi là hợp lệ.

In [ ]:
REPO = "trannhiem/TranNhiem-Vietnamese-DocumentImage-Reasoning"
jsonl = hf_hub_download(REPO, "data/vdoc.jsonl", repo_type="dataset", local_dir=DATA_DIR)
for s in range(N_SHARDS):
    marker = f"{DATA_DIR}/shards/images-{s:05d}.done"
    if os.path.exists(marker):
        continue
    tar_path = hf_hub_download(REPO, f"shards/images-{s:05d}.tar", repo_type="dataset", local_dir=DATA_DIR)
    with tarfile.open(tar_path) as t:
        t.extractall(DATA_DIR, filter="data")
    os.remove(tar_path)
    open(marker, "w").close()

records, skipped = [], 0
with open(jsonl, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        path = os.path.join(DATA_DIR, r["image"])
        if not os.path.exists(path):
            continue
        if not str(r["id"]).isdigit():
            skipped += 1
            continue
        qs = [t["value"].strip() for t in r["conversations"]
              if t["from"] == "human" and isinstance(t["value"], str) and len(t["value"].strip()) > 10]
        if qs:
            records.append({"id": int(r["id"]), "path": path, "qs": qs})
print(f"{len(records)} trang V-Doc hợp lệ (bỏ {skipped} trang có id không phải số)")

def img_hash_pil(im):
    return hashlib.md5(im.convert("L").resize((32, 32)).tobytes()).hexdigest()

def img_hash(p):
    with Image.open(p) as im:
        im.draft("L", (64, 64))
        return img_hash_pil(im)

for r in tqdm(records, desc="hash ảnh V-Doc"):
    r["h"] = img_hash(r["path"])

records.sort(key=lambda r: r["id"])
is_test = lambda r: (r["id"] // BLOCK) % TEST_EVERY == 0
test_pages, test_h = [], set()
for r in records:
    if is_test(r) and r["h"] not in test_h:
        test_h.add(r["h"])
        test_pages.append(r)
random.Random(SEED).shuffle(test_pages)
test_pages = test_pages[:MAX_TEST]
test_h = {r["h"] for r in test_pages}
train_pages = [r for r in records if not is_test(r) and r["h"] not in test_h]

TEST_DOCS = [r["path"] for r in test_pages]
TEST_Q = [r["qs"][0] for r in test_pages]
TEST_T = list(range(len(test_pages)))

rng = random.Random(SEED)
is_val = lambda r: (r["id"] // BLOCK) % TEST_EVERY == 1
val_pages, val_h = [], set()
for r in train_pages:
    if is_val(r) and r["h"] not in val_h:
        val_h.add(r["h"])
        val_pages.append(r)
rng.shuffle(val_pages)
val_pages = val_pages[:N_VAL]
val_h = {r["h"] for r in val_pages}

sg_pages = [r for r in train_pages if not is_val(r) and r["h"] not in val_h]
rng.shuffle(sg_pages)
sg_pages = sg_pages[:N_TRAIN_PAGES]
tr_q, tr_t = [], []
for i, r in enumerate(sg_pages):
    for q in r["qs"][:MAX_Q_PER_PAGE]:
        tr_q.append(q)
        tr_t.append(i)
VAL_T = list(range(len(val_pages)))

vdoc_tr_h = {r["h"] for r in sg_pages}
assert vdoc_tr_h.isdisjoint(val_h) and vdoc_tr_h.isdisjoint(test_h) and val_h.isdisjoint(test_h), "Rò rỉ hash V-Doc!"
assert {r["id"] for r in train_pages}.isdisjoint({r["id"] for r in test_pages}), "Rò rỉ id V-Doc!"

# Bộ kiểm tra tiếng Anh: ViDoRe DocVQA
EN_DOCS, EN_Q, EN_T = [], [], []
en_h = set()
if RUN_EN_CHECK:
    idx = {}
    os.makedirs(f"{DATA_DIR}/en", exist_ok=True)
    for row in load_dataset("vidore/docvqa_test_subsampled", split="test"):
        if row["image_filename"] not in idx:
            p = f"{DATA_DIR}/en/{len(EN_DOCS)}.png"
            row["image"].save(p)
            idx[row["image_filename"]] = len(EN_DOCS)
            EN_DOCS.append(p)
            en_h.add(img_hash(p))
        if row["query"]:
            EN_Q.append(row["query"])
            EN_T.append(idx[row["image_filename"]])

norm_q = lambda s: re.sub(r"\s+", " ", str(s).strip().lower())
forbidden_q_norm = {norm_q(q) for q in (TEST_Q + EN_Q)}
forbidden_hashes = test_h | val_h | en_h | vdoc_tr_h

print(f"V-Doc -> Train: {len(sg_pages)} trang ({len(tr_q)} câu) | Val: {len(val_pages)} | Test: {len(TEST_Q)} trang")
print(f"EN DocVQA -> {len(EN_Q)} câu / {len(EN_DOCS)} trang | Tập hash cấm leakage: {len(forbidden_hashes)}")
print("Khớp TEST_Q[0]:", TEST_Q[0])

## 2. Hàm đánh giá (giữ nguyên Giai đoạn 2) + thống kê chuẩn hoá
- Giữ nguyên `score_ragged_maxsim`, `ranks_from_scores` (xếp hạng bi quan), `per_query`, `metrics`.
- **Mới:** randomization test có cặp hai phía, `p = (b+1)/(B+1)` (không bao giờ in `0.000`), hiệu chỉnh **Holm** cho cả họ so sánh, CI95 bootstrap cho cả chênh lệch, điểm từng câu được trung bình qua seed trước khi kiểm định.
- Kết quả ghi vào file JSON **riêng của Giai đoạn 3**; file Giai đoạn 2 chỉ được đọc để đối chiếu.

In [ ]:
KEYS = ["nDCG@5", "R@1", "R@5", "MRR@10"]
MB_PER_1K = lambda vec, dim=128: vec * dim * 2 * 1000 / 2**20

def load_img(x):
    return (x if isinstance(x, Image.Image) else Image.open(x)).convert("RGB")

def to_dev(batch, model, cast_mask=False):
    p = next(model.parameters())
    dev, dt = p.device, p.dtype
    out = {}
    for k, v in batch.items():
        if k == "labels":
            continue
        if not torch.is_tensor(v):
            out[k] = v
            continue
        is_float = v.is_floating_point() or (cast_mask and k == "attention_mask")
        out[k] = v.to(dev, dtype=dt) if is_float else v.to(dev)
    return out

@torch.no_grad()
def encode(model, proc, items, is_image, bs=4, cast_mask=False):
    embs = []
    for i in tqdm(range(0, len(items), bs), desc="pages" if is_image else "queries", leave=False):
        chunk = items[i:i + bs]
        b = proc.process_images([load_img(x) for x in chunk]) if is_image else proc.process_queries(chunk)
        b = to_dev(b, model, cast_mask=cast_mask)
        e = model(**b)
        if isinstance(e, (tuple, list)):
            e = e[0]
        elif hasattr(e, "embeddings"):
            e = e.embeddings
        has_mask = ("attention_mask" in b) and torch.is_tensor(b["attention_mask"]) and (e.shape[1] == b["attention_mask"].shape[1])
        if has_mask:
            for x, m in zip(e, b["attention_mask"]):
                embs.append(x[m.bool()].to("cpu", torch.bfloat16))
        else:
            for x in e:
                valid = x.float().norm(dim=-1) > 1e-4
                embs.append((x[valid] if valid.any() else x).to("cpu", torch.bfloat16))
    return embs

def _pad_with_mask(seq_list):
    lens = [x.shape[0] for x in seq_list]
    padded = pad_sequence([x.float() for x in seq_list], batch_first=True)
    idx = torch.arange(padded.shape[1])[None, :]
    mask = idx < torch.tensor(lens)[:, None]
    return padded, mask

@torch.no_grad()
def score_ragged_maxsim(Q_list, D_list, G_list=None, q_chunk=32, d_chunk=128, device=DEVICE):
    """Tính MaxSim cho danh sách tensor độ dài động, có hỗ trợ trọng số cổng G_list."""
    use_cuda = str(device).startswith("cuda") and torch.cuda.is_available()
    comp_dt = torch.bfloat16 if (use_cuda and torch.cuda.is_bf16_supported()) else (torch.float16 if use_cuda else torch.float32)
    out = torch.empty(len(Q_list), len(D_list), dtype=torch.float32)
    for i in range(0, len(Q_list), q_chunk):
        q_pad, q_mask = _pad_with_mask(Q_list[i:i + q_chunk])
        q_dev = q_pad.to(device, dtype=comp_dt)
        qm_dev = q_mask.to(device)
        if G_list is not None:
            g_pad = pad_sequence([g.float() for g in G_list[i:i + q_chunk]], batch_first=True).to(device)
            g_eff = g_pad * qm_dev.float()
        else:
            g_eff = qm_dev.float()
        for j in range(0, len(D_list), d_chunk):
            d_pad, d_mask = _pad_with_mask(D_list[j:j + d_chunk])
            d_dev = d_pad.to(device, dtype=comp_dt)
            dm_dev = d_mask.to(device)
            sim = torch.einsum("bqd,cpd->bcqp", q_dev, d_dev).float()
            sim = sim.masked_fill(~dm_dev[None, :, None, :], -1e9)
            max_sim = sim.amax(dim=-1) * g_eff[:, None, :]
            out[i:i + q_chunk, j:j + d_chunk] = max_sim.sum(dim=-1).cpu()
    return out

def ranks_from_scores(S, targets):
    S = torch.as_tensor(S, dtype=torch.float32)
    t = torch.as_tensor(targets)
    pos = S[torch.arange(len(t)), t]
    return (S >= pos[:, None]).sum(1).numpy()

def per_query(r, key):
    r = np.asarray(r, dtype=float)
    return {
        "nDCG@5": np.where(r <= 5, 1 / np.log2(r + 1), 0),
        "R@1": (r <= 1) * 1.0,
        "R@5": (r <= 5) * 1.0,
        "MRR@10": np.where(r <= 10, 1 / r, 0),
    }[key] * 100

def metrics(r):
    return {k: per_query(r, k).mean() for k in KEYS}

def boot(v, n=2000):
    return v[np.random.default_rng(0).integers(0, len(v), (n, len(v)))].mean(1)

def ci(r, key="nDCG@5"):
    return np.percentile(boot(per_query(r, key)), [2.5, 97.5])

def paired_p(ra, rb, key="nDCG@5"):
    b = boot(per_query(ra, key) - per_query(rb, key))
    return min(1.0, 2 * min((b <= 0).mean(), (b >= 0).mean()))

def vec_per_page(D):
    return float(np.mean([len(x) for x in D]))

def free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def sync_ms(fn, n):
    use_cuda = torch.cuda.is_available()
    fn()
    if use_cuda:
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = fn()
    if use_cuda:
        torch.cuda.synchronize()
    return out, (time.perf_counter() - t0) * 1000 / n

# ---- Unit tests nhanh cho score_ragged_maxsim (có và không có G_list) ----
_rng = torch.Generator().manual_seed(0)
_q_rag = [F.normalize(torch.randn(3, 16, generator=_rng), dim=-1), F.normalize(torch.randn(7, 16, generator=_rng), dim=-1)]
_d_rag = [F.normalize(torch.randn(5, 16, generator=_rng), dim=-1), F.normalize(torch.randn(11, 16, generator=_rng), dim=-1)]
_g_rag = [torch.tensor([0.5, 1.8, 0.7]), torch.linspace(0.6, 1.4, 7)]
_s_no_g = score_ragged_maxsim(_q_rag, _d_rag, G_list=None, q_chunk=1, d_chunk=1, device="cpu")
_s_no_g_ref = torch.tensor([[(q @ d.T).max(dim=-1).values.sum().item() for d in _d_rag] for q in _q_rag])
assert torch.allclose(_s_no_g, _s_no_g_ref, atol=1e-3)
_s_with_g = score_ragged_maxsim(_q_rag, _d_rag, G_list=_g_rag, q_chunk=1, d_chunk=1, device="cpu")
_s_with_g_ref = torch.tensor([[((q @ d.T).max(dim=-1).values * g).sum().item() for d in _d_rag] for q, g in zip(_q_rag, _g_rag)])
assert torch.allclose(_s_with_g, _s_with_g_ref, atol=1e-3)

# ---- Nạp RESULTS của notebook này (file JSON RIÊNG của Giai đoạn 3; không kế thừa và không ghi đè file Giai đoạn 1–2) ----
RESULTS = json.load(open(RES_PATH, encoding="utf-8")) if os.path.exists(RES_PATH) else {}
RESULTS_V2 = json.load(open(RES_V2_PATH, encoding="utf-8")) if os.path.exists(RES_V2_PATH) else {}
print(f"Kết quả Giai đoạn 2 (chỉ đọc, để đối chiếu): {len(RESULTS_V2)} mốc từ {RES_V2_PATH}")

def save_result(name, **kw):
    RESULTS[name] = {k: (None if v is None else np.asarray(v).tolist()) for k, v in kw.items()}
    with open(RES_PATH, "w", encoding="utf-8") as f:
        json.dump(RESULTS, f, ensure_ascii=False)

ocr_path = f"{SAVE_DIR}/ocr_test_{len(TEST_DOCS)}.json"
OCR_TEXTS = json.load(open(ocr_path, encoding="utf-8")) if os.path.exists(ocr_path) else None
print(f"Đã nạp {len(RESULTS)} mốc kết quả sẵn có:", list(RESULTS))

# =====================================================================================
# Thống kê chuẩn hoá cho IR (Giai đoạn 3)
#  - Điểm từng câu hỏi được TRUNG BÌNH QUA CÁC SEED trước, rồi mới kiểm định (không gộp seed×query thành mẫu độc lập)
#  - Paired randomization (sign-flip) test hai phía: p = (b+1)/(B+1) -> không bao giờ ra 0
#  - Holm–Bonferroni cho toàn bộ họ phép so sánh trong bảng
#  - CI95 bootstrap theo câu hỏi cho trung bình và cho CHÊNH LỆCH
# =====================================================================================
def pq_mean(v, key="nDCG@5"):
    """v: ranks 1 seed (list[int]) hoặc nhiều seed (list[list[int]]) -> điểm từng câu hỏi trung bình qua seed."""
    return np.mean([per_query(r, key) for r in np.atleast_2d(np.asarray(v, dtype=float))], 0)

def seed_scores(v, key="nDCG@5"):
    return [float(per_query(r, key).mean()) for r in np.atleast_2d(np.asarray(v, dtype=float))]

def paired_randomization_p(a, b, n_perm=None, seed=0):
    n_perm = N_PERM if n_perm is None else n_perm
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    n = len(d)
    obs = abs(d.mean())
    if n == 0 or obs < 1e-12:
        return 1.0
    rng = np.random.default_rng(seed)
    hits, done = 0, 0
    while done < n_perm:
        m = min(10000, n_perm - done)
        signs = rng.integers(0, 2, size=(m, n), dtype=np.int8).astype(np.float64) * 2.0 - 1.0
        hits += int((np.abs(signs @ d) / n >= obs - 1e-12).sum())
        done += m
    return (hits + 1) / (n_perm + 1)

def holm_adjust(pvals):
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    idx = np.where(ok)[0]
    if len(idx) == 0:
        return out
    order = idx[np.argsort(p[idx])]
    m, running = len(order), 0.0
    for k, i in enumerate(order):
        running = max(running, (m - k) * p[i])
        out[i] = min(1.0, running)
    return out

def boot_ci(v, n=None, seed=0):
    n = N_BOOT if n is None else n
    v = np.asarray(v, dtype=float)
    bs = v[np.random.default_rng(seed).integers(0, len(v), (n, len(v)))].mean(1)
    return np.percentile(bs, [2.5, 97.5])

def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return None
    return f"{p:.1e}" if p < 1e-3 else f"{p:.4f}"

# ---- Tự kiểm tra nhanh (chạy trên CPU, vài giây) ----
_a = np.r_[np.ones(200), np.zeros(100)] * 100
_b = np.r_[np.ones(100), np.zeros(200)] * 100
assert paired_randomization_p(_a, _b, n_perm=2000) <= 1 / 2001 + 1e-9
assert paired_randomization_p(_a, _a, n_perm=2000) == 1.0
_c = np.tile([100.0, 0.0, 100.0, 0.0, 0.0, 100.0], 30)
assert 0.05 < paired_randomization_p(_c, np.roll(_c, 1), n_perm=2000) <= 1.0  # chênh lệch nhỏ -> không có ý nghĩa
_h = holm_adjust([0.01, 0.04, np.nan, 0.03])
assert np.allclose(_h[[0, 1, 3]], [0.03, 0.06, 0.06]) and np.isnan(_h[2])
print("✓ Tự kiểm tra thống kê (randomization test + Holm) OK")

## 3. Kiến trúc & vòng train Giai đoạn 2 (chép nguyên văn, không sửa)
Cần cho các dòng đối chứng: nhánh query của v2 (baseline) và kiến trúc v2 chạy lại với `'Query: '`.

In [ ]:
def score_masked_maxsim(q_emb, q_mask, d_emb, d_mask, gates=None, q_chunk_size=8, d_chunk_size=64):
    B = q_emb.shape[0]
    C = d_emb.shape[0]
    q_fl, d_fl = q_emb.float(), d_emb.float()
    g_eff = (gates.float() * q_mask.float()) if gates is not None else q_mask.float()
    score_rows = []
    for i in range(0, B, q_chunk_size):
        q_sub = q_fl[i:i + q_chunk_size]
        g_sub = g_eff[i:i + q_chunk_size].unsqueeze(1)
        doc_cols = []
        for j in range(0, C, d_chunk_size):
            d_sub = d_fl[j:j + d_chunk_size]
            dm_sub = d_mask[j:j + d_chunk_size]
            sim = torch.einsum("bqd,cpd->bcqp", q_sub, d_sub)
            sim = sim.masked_fill(~dm_sub[None, :, None, :], -1e9)
            max_sim = sim.max(dim=-1).values
            doc_cols.append((max_sim * g_sub).sum(dim=-1))
        score_rows.append(torch.cat(doc_cols, dim=1))
    return torch.cat(score_rows, dim=0)

class DynamicMultiVectorAdapter(nn.Module):
    def __init__(self, dim=128, rank=32, window_size=8, dropout=0.1,
                 doc_mode="dyn_merge", use_doc_adapter=True,
                 use_subword_attn=True, use_query_gate=True):
        super().__init__()
        assert doc_mode in ("none", "dyn_merge")
        assert window_size == 8, "2 vòng gộp đôi 8 -> 4 -> 2 yêu cầu window_size=8"
        self.dim = dim
        self.window_size = window_size
        self.doc_mode = doc_mode
        self.use_doc_adapter = use_doc_adapter
        self.use_subword_attn = use_subword_attn
        self.use_query_gate = use_query_gate

        # Nhánh 1A: Masked 1D Context Conv & Position-Free Saliency
        self.doc_conv = nn.Sequential(
            nn.Conv1d(dim, dim, kernel_size=3, padding=1, groups=dim, bias=False),
            nn.GroupNorm(8, dim),
            nn.GELU(),
            nn.Conv1d(dim, rank, kernel_size=1, bias=False),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(rank, dim, kernel_size=1, bias=False),
        )
        self.alpha_conv = nn.Parameter(torch.tensor(0.02))

        self.saliency_head = nn.Sequential(
            nn.Conv1d(3, 16, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(16, 1, kernel_size=1),
        )
        nn.init.normal_(self.saliency_head[-1].weight, std=0.02)
        nn.init.zeros_(self.saliency_head[-1].bias)
        self.saliency_temp = nn.Parameter(torch.tensor(0.30))

        # Nhánh 1B: Window Cross-Attention Refiner
        self.merge_norm = nn.LayerNorm(dim)
        self.merge_q = nn.Linear(dim, rank, bias=False)
        self.merge_k = nn.Linear(dim, rank, bias=False)
        self.merge_v = nn.Linear(dim, rank, bias=False)
        self.merge_out = nn.Linear(rank, dim, bias=False)
        self.alpha_merge = nn.Parameter(torch.tensor(0.04))

        # Nhánh 2A: Low-Rank Bidirectional Subword Attention
        self.sub_norm = nn.LayerNorm(dim)
        self.sub_q = nn.Linear(dim, rank, bias=False)
        self.sub_k = nn.Linear(dim, rank, bias=False)
        self.sub_v = nn.Linear(dim, rank, bias=False)
        self.sub_out = nn.Linear(rank, dim, bias=False)
        self.alpha_sub = nn.Parameter(torch.tensor(0.04))

        # Nhánh 2B: Position-Free Query Context Gate (5 đặc trưng, ĐÃ BỎ f6_pos)
        self.gate_mlp = nn.Sequential(
            nn.Linear(dim + 5, 48),
            nn.LayerNorm(48),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(48, 1),
        )
        nn.init.normal_(self.gate_mlp[-1].weight, std=0.01)
        nn.init.zeros_(self.gate_mlp[-1].bias)
        self.gate_scale = nn.Parameter(torch.tensor(0.30))

    def _merge_windows(self, win_x, win_w, win_m):
        """Gộp 2 vòng trong từng cửa sổ độ dài 8: 8 -> 4 -> 2, không để padding lọt vào."""
        B, S, _, D = win_x.shape
        x = win_x
        w = win_w * win_m.float()
        m = win_m
        for _ in range(2):
            A, wA, mA = x[:, :, 0::2, :], w[:, :, 0::2], m[:, :, 0::2]
            B_set, wB, mB = x[:, :, 1::2, :], w[:, :, 1::2], m[:, :, 1::2]
            sim = torch.einsum("bsid,bsjd->bsij", B_set, A)
            has_valid_A = mA.any(dim=-1, keepdim=True)
            sim = sim.masked_fill((~mA.unsqueeze(2)) & has_valid_A.unsqueeze(2), -1e9)
            best_a = sim.argmax(dim=-1)

            merged_A = A * wA.unsqueeze(-1)
            merged_A.scatter_add_(2, best_a.unsqueeze(-1).expand(-1, -1, -1, D), B_set * wB.unsqueeze(-1))

            new_w = wA.clone()
            new_w.scatter_add_(2, best_a, wB)
            new_m = new_w > 1e-6
            x = F.normalize(merged_A, p=2, dim=-1) * new_m.float().unsqueeze(-1)
            w = new_w
            m = new_m
        return x, m

    def forward_docs(self, E_d, d_mask):
        d_mask_b = d_mask.bool()
        d_mask_fl = d_mask_b.float()
        x = F.normalize(E_d.float(), p=2, dim=-1) * d_mask_fl.unsqueeze(-1)

        if self.use_doc_adapter:
            x_b = x.permute(0, 2, 1)
            m_b = d_mask_fl.unsqueeze(1)
            num_d = d_mask_fl.sum(dim=1, keepdim=True).clamp(min=1.0)

            sum_local = F.avg_pool1d(x_b, kernel_size=3, stride=1, padding=1) * 3.0
            cnt_local = F.avg_pool1d(m_b, kernel_size=3, stride=1, padding=1) * 3.0
            mean_local = sum_local / cnt_local.clamp(min=1.0)
            diff_local3 = (x_b - mean_local).norm(dim=1, keepdim=True) * m_b

            doc_mean = x.sum(dim=1, keepdim=True) / num_d.unsqueeze(-1)
            diff_glob = (x - doc_mean).norm(dim=-1).unsqueeze(1) * m_b
            max_ch = x_b.abs().max(dim=1, keepdim=True).values * m_b

            sal_raw = torch.cat([diff_local3, diff_glob, max_ch], dim=1)
            sal_mean = (sal_raw * m_b).sum(dim=2, keepdim=True) / num_d.unsqueeze(-1)
            sal_var = (((sal_raw - sal_mean) ** 2) * m_b).sum(dim=2, keepdim=True) / num_d.unsqueeze(-1)
            sal_in = ((sal_raw - sal_mean) / (sal_var.sqrt() + 1e-5)) * m_b
            sal_logits = self.saliency_head(sal_in).squeeze(1) * d_mask_fl
            patch_weights = torch.exp(torch.clamp(self.saliency_temp, 0.05, 0.8) * torch.tanh(sal_logits)) * d_mask_fl

            conv_delta = self.doc_conv(x_b).permute(0, 2, 1) * d_mask_fl.unsqueeze(-1)
            x = F.normalize(x + 0.15 * torch.tanh(self.alpha_conv) * conv_delta, p=2, dim=-1) * d_mask_fl.unsqueeze(-1)
        else:
            sal_logits = None
            patch_weights = d_mask_fl

        if self.doc_mode == "none":
            return x, d_mask_b

        B, L, D = x.shape
        W = self.window_size
        pad_len = (W - (L % W)) % W
        if pad_len > 0:
            x = F.pad(x, (0, 0, 0, pad_len))
            patch_weights = F.pad(patch_weights, (0, pad_len))
            d_mask_b = F.pad(d_mask_b, (0, pad_len), value=False)
            if sal_logits is not None:
                sal_logits = F.pad(sal_logits, (0, pad_len))

        S = (L + pad_len) // W
        win_x = x.view(B, S, W, D)
        win_w = patch_weights.view(B, S, W)
        win_m = d_mask_b.view(B, S, W)

        seeds, seed_mask = self._merge_windows(win_x, win_w, win_m)

        if self.use_doc_adapter:
            win_sal = sal_logits.view(B, S, W)
            sn = self.merge_norm(win_x)
            qn = self.merge_norm(seeds)
            Q, K, V = self.merge_q(qn), self.merge_k(sn), self.merge_v(sn)
            attn_logits = torch.einsum("bsqr,bskr->bsqk", Q, K) / (Q.shape[-1] ** 0.5) + 0.3 * torch.tanh(win_sal).unsqueeze(2)
            attn_logits = attn_logits.masked_fill(~win_m.unsqueeze(2), -1e4)
            attn_weights = F.softmax(attn_logits, dim=-1) * win_m.float().unsqueeze(2)
            attn_out = self.merge_out(torch.einsum("bsqk,bskr->bsqr", attn_weights, V))
            merged = F.normalize(seeds + 0.25 * torch.tanh(self.alpha_merge) * attn_out, p=2, dim=-1) * seed_mask.float().unsqueeze(-1)
        else:
            merged = seeds

        return merged.reshape(B, S * 2, D), seed_mask.reshape(B, S * 2)

    def forward_queries(self, E_q, q_mask):
        q_mask_b = q_mask.bool()
        valid_float = q_mask_b.float()
        q_float = F.normalize(E_q.float(), p=2, dim=-1) * valid_float.unsqueeze(-1)

        if self.use_subword_attn:
            qn = self.sub_norm(q_float)
            Q, K, V = self.sub_q(qn), self.sub_k(qn), self.sub_v(qn)
            scores = torch.einsum("bqr,bkr->bqk", Q, K) / (Q.shape[-1] ** 0.5)
            scores = scores.masked_fill(~q_mask_b.unsqueeze(1), -1e4)
            attn_w = F.softmax(scores, dim=-1) * valid_float.unsqueeze(-1)
            sub_delta = self.sub_out(torch.einsum("bqk,bkr->bqr", attn_w, V))
            q_contextual = F.normalize(q_float + 0.20 * torch.tanh(self.alpha_sub) * sub_delta, p=2, dim=-1) * valid_float.unsqueeze(-1)
        else:
            q_contextual = q_float

        if self.use_query_gate:
            B, L, _ = q_contextual.shape
            num_valid = valid_float.sum(dim=-1, keepdim=True).clamp(min=1.0)
            q_global = F.normalize(q_contextual.sum(dim=1, keepdim=True) / num_valid.unsqueeze(-1), p=2, dim=-1)

            f1_dist = 1.0 - (q_contextual * q_global).sum(dim=-1, keepdim=True)
            prev_q = torch.roll(q_contextual, shifts=1, dims=1); prev_q[:, 0, :] = 0.0
            next_q = torch.roll(q_contextual, shifts=-1, dims=1); next_q[:, -1, :] = 0.0
            f2_prev = (q_contextual * prev_q).sum(dim=-1, keepdim=True)
            f3_next = (q_contextual * next_q).sum(dim=-1, keepdim=True)

            sim_mat = torch.einsum("bqd,bkd->bqk", q_contextual, q_contextual)
            eye_mask = torch.eye(L, device=E_q.device).bool().unsqueeze(0)
            peer_mask = q_mask_b.unsqueeze(1) & (~eye_mask)
            f4_mean = sim_mat.masked_fill(~peer_mask, 0.0).sum(dim=-1, keepdim=True) / (num_valid.unsqueeze(-1) - 1.0).clamp(min=1.0)
            f5_max = sim_mat.masked_fill(~peer_mask, -1.0).max(dim=-1, keepdim=True).values.clamp(min=0.0)

            gate_in = torch.cat([q_contextual, f1_dist, f2_prev, f3_next, f4_mean, f5_max], dim=-1) * valid_float.unsqueeze(-1)
            gate_delta = torch.tanh(self.gate_mlp(gate_in)).squeeze(-1) * valid_float
            scale = torch.clamp(self.gate_scale, 0.05, 0.50)
            raw_gates = (1.0 + scale * gate_delta) * valid_float
            gates_norm = raw_gates * (num_valid / (raw_gates.sum(dim=-1, keepdim=True) + 1e-6))
            return q_contextual, gates_norm, gate_delta, q_float
        else:
            return q_contextual, None, torch.zeros(1, device=E_q.device), q_float

    def forward(self, E_q, q_mask, E_d, d_mask):
        if self.use_doc_adapter:
            d_emb, out_d_mask = self.forward_docs(E_d, d_mask)
        else:
            with torch.no_grad():
                d_emb, out_d_mask = self.forward_docs(E_d, d_mask)

        q_cs = 4 if self.doc_mode == "none" else SCORE_CHUNK
        if self.use_subword_attn or self.use_query_gate:
            q_emb, gates_norm, gate_delta, q_orig = self.forward_queries(E_q, q_mask)
            scores = score_masked_maxsim(q_emb, q_mask, d_emb, out_d_mask, gates=gates_norm,
                                         q_chunk_size=q_cs, d_chunk_size=DOC_CHUNK)
            return scores, gate_delta, q_emb, q_orig
        else:
            q_orig = F.normalize(E_q.float(), p=2, dim=-1) * q_mask.float().unsqueeze(-1)
            scores = score_masked_maxsim(q_orig, q_mask, d_emb, out_d_mask, gates=None,
                                         q_chunk_size=q_cs, d_chunk_size=DOC_CHUNK)
            return scores, torch.zeros(1, device=E_q.device), q_orig, q_orig

_tmp_net = DynamicMultiVectorAdapter(dim=128, rank=32)
print(f"Tổng tham số DynamicMultiVectorAdapter: {sum(p.numel() for p in _tmp_net.parameters()):,} (~{sum(p.numel() for p in _tmp_net.parameters()) * 4 / 1024:.1f} KB)")
del _tmp_net

In [ ]:
@torch.no_grad()
def eval_adapter_ragged(net, Q_list, D_list, targets, doc_bs=32, q_bs=64):
    net.eval()
    D_comp = []
    for i in range(0, len(D_list), doc_bs):
        d_pad, d_mask = _pad_with_mask(D_list[i:i + doc_bs])
        d_out, m_out = net.forward_docs(d_pad.to(DEVICE), d_mask.to(DEVICE))
        d_out_cpu = d_out.to("cpu", torch.bfloat16)
        m_out_cpu = m_out.cpu()
        for x, m in zip(d_out_cpu, m_out_cpu):
            D_comp.append(x[m])

    Q_adapt, G_adapt = [], []
    use_g = net.use_query_gate
    for i in range(0, len(Q_list), q_bs):
        q_pad, q_mask = _pad_with_mask(Q_list[i:i + q_bs])
        q_out, g_out, _, _ = net.forward_queries(q_pad.to(DEVICE), q_mask.to(DEVICE))
        q_out_cpu = q_out.to("cpu", torch.bfloat16)
        q_mask_cpu = q_mask.cpu()
        g_cpu = g_out.float().cpu() if (use_g and g_out is not None) else None
        for idx_q in range(len(q_out_cpu)):
            m = q_mask_cpu[idx_q]
            Q_adapt.append(q_out_cpu[idx_q][m])
            if g_cpu is not None:
                G_adapt.append(g_cpu[idx_q][m])

    S = score_ragged_maxsim(Q_adapt, D_comp, G_list=G_adapt if use_g else None)
    return ranks_from_scores(S, targets), D_comp

@torch.no_grad()
def time_doc_compression(net, D_list, doc_bs=32):
    net.eval()
    batches = []
    for i in range(0, len(D_list), doc_bs):
        d_pad, d_mask = _pad_with_mask(D_list[i:i + doc_bs])
        batches.append((d_pad.to(DEVICE), d_mask.to(DEVICE)))
    _, ms = sync_ms(lambda: [net.forward_docs(xb, mb) for xb, mb in batches], len(D_list))
    del batches
    return ms

def mine_hard_negatives(Q_tr_list, D_tr_list, tr_t_np, tr_h_ids, k=HARD_NEG):
    S_tea = score_ragged_maxsim(Q_tr_list, D_tr_list)
    print("ColQwen2.5 gốc trên tập train:",
          {m: round(float(v), 1) for m, v in metrics(ranks_from_scores(S_tea, tr_t_np)).items()})
    same = tr_h_ids[None, :] == tr_h_ids[tr_t_np][:, None]
    hard = S_tea.masked_fill(torch.as_tensor(same), -1e9).topk(k, dim=1).indices.numpy()
    del S_tea, same
    return hard

def train_qwen_adapter(Q_tr_list, D_tr_list, tr_t_np, tr_h_ids, hard_neg_np,
                       Q_val_list, D_val_list, doc_mode, use_doc_adapter,
                       use_subword_attn, use_query_gate, use_kd, seed):
    torch.manual_seed(seed)
    gen = np.random.default_rng(seed)
    emb_dim = int(D_tr_list[0].shape[-1])
    net = DynamicMultiVectorAdapter(
        dim=emb_dim, rank=32, window_size=8, dropout=0.1,
        doc_mode=doc_mode, use_doc_adapter=use_doc_adapter,
        use_subword_attn=use_subword_attn, use_query_gate=use_query_gate,
    ).to(DEVICE)

    Nq = len(Q_tr_list)
    steps_per_epoch = int(np.ceil(Nq / BQ))
    total_steps = steps_per_epoch * EPOCHS
    warmup_steps = steps_per_epoch * WARMUP_EPOCHS

    opt = torch.optim.AdamW(net.parameters(), lr=LR_ADAPTER, weight_decay=WEIGHT_DECAY)
    sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    best, best_state = -1.0, None

    for ep in range(EPOCHS):
        net.train()
        perm = gen.permutation(Nq)
        ep_loss, n_steps = 0.0, 0
        for i in range(0, Nq, BQ):
            idx = perm[i:i + BQ]
            pos = tr_t_np[idx]
            cand = np.unique(np.concatenate([pos, hard_neg_np[idx].reshape(-1)]))
            tgt = torch.as_tensor(np.searchsorted(cand, pos), dtype=torch.long, device=DEVICE)

            pos_h = tr_h_ids[pos][:, None]
            cand_h = tr_h_ids[cand][None, :]
            fn = torch.as_tensor((cand_h == pos_h) & (cand[None, :] != pos[:, None]), device=DEVICE)

            q_pad, q_mask = _pad_with_mask([Q_tr_list[j] for j in idx])
            d_pad, d_mask = _pad_with_mask([D_tr_list[j] for j in cand])
            q_pad, q_mask = q_pad.to(DEVICE), q_mask.to(DEVICE)
            d_pad, d_mask = d_pad.to(DEVICE), d_mask.to(DEVICE)

            s, g_delta, q_emb, q_orig = net(q_pad, q_mask, d_pad, d_mask)
            s = s.masked_fill(fn, -1e4)
            loss = F.cross_entropy(s / TEMP_CE, tgt)

            if use_kd:
                with torch.no_grad():
                    d_tea = F.normalize(d_pad.float(), p=2, dim=-1) * d_mask.float().unsqueeze(-1)
                    t = score_masked_maxsim(
                        q_orig, q_mask, d_tea, d_mask, gates=None, q_chunk_size=SCORE_CHUNK, d_chunk_size=DOC_CHUNK
                    ).masked_fill(fn, -1e4)
                kd = F.kl_div(F.log_softmax(s / TEMP_KD, -1), F.softmax(t / TEMP_KD, -1), reduction="batchmean") * (TEMP_KD ** 2)
                anchor = (1.0 - (q_emb * q_orig).sum(dim=-1))[q_mask].mean()
                loss = 0.50 * loss + 0.50 * kd + ANCHOR_W * anchor

            if use_query_gate:
                loss = loss + 0.005 * g_delta.abs().mean()

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
            sched.step()
            ep_loss += float(loss.item())
            n_steps += 1

        val_ranks, _ = eval_adapter_ragged(net, Q_val_list, D_val_list, VAL_T)
        nd = float(metrics(val_ranks)["nDCG@5"])
        if nd > best:
            best, best_state = nd, copy.deepcopy(net.state_dict())
        print(f"  epoch {ep + 1}/{EPOCHS}  loss {ep_loss / max(1, n_steps):.4f}  val nDCG@5 {nd:.1f}")

    net.load_state_dict(best_state)
    return net.eval()

## 4. Cache embedding + **sửa giao thức query** (`'Query: '`) + bộ test ngoài miền
- `colpali-engine` ≥ 0.3.13 đặt `query_prefix = ""`, trong khi `vidore/colqwen2.5-v0.2` được train với `"Query: " + câu hỏi + 10×<|endoftext|>` → Giai đoạn 2 đo backbone **dưới giao thức sai**. Ở đây chỉ encode lại **câu hỏi** (embedding trang giữ nguyên cache Giai đoạn 2).
- Lưu thêm `content mask` cho từng câu (token nào thuộc câu hỏi, token nào là tiền tố/augmentation) để phân tích gate đúng.
- **OOD:** `ViInfographicVQA` single-image test (193 infographic thật / 1.374 câu hỏi do người kiểm duyệt) — khác miền sách giáo khoa và khác nguồn sinh câu hỏi của V-Doc.

In [ ]:
def load_model_safe(loader_fn, proc, sample_imgs, sample_qs, cast_mask=False):
    if torch.cuda.is_available():
        if PREFER_FP16_ON_T4 and not torch.cuda.is_bf16_supported():
            dtypes = [torch.float16, torch.bfloat16]
        else:
            dtypes = [torch.bfloat16]
    else:
        dtypes = [torch.float32]

    last_err = None
    for dt in dtypes:
        m = None
        try:
            print(f"  Thử nạp model với dtype={dt}...")
            m = loader_fn(dt).eval()
            out_d = encode(m, proc, sample_imgs[:2], True, bs=2, cast_mask=cast_mask)
            out_q = encode(m, proc, sample_qs[:2], False, bs=2, cast_mask=cast_mask)
            if all(torch.isfinite(x).all() and x.float().norm(dim=-1).mean() > 1e-3 for x in out_d + out_q):
                print(f"  ✓ Sanity check thành công với dtype={dt} | shape trang mẫu: {tuple(out_d[0].shape)}")
                return m, dt
            raise RuntimeError(f"Sanity check phát hiện NaN/Inf hoặc vector 0 với dtype={dt}")
        except Exception as exc:
            last_err = exc
            print(f"  ✗ Lỗi với dtype={dt}: {exc}")
            if m is not None:
                del m
            free()
    raise RuntimeError(f"Không nạp được model. Lỗi cuối: {last_err}")
# =====================================================================================
# Cache embedding ColQwen2.5 (tái sử dụng cache Giai đoạn 2) + encode lại QUERY có tiền tố "Query: " + bộ OOD
# =====================================================================================
from colpali_engine.models import ColQwen2_5, ColQwen2_5_Processor

PROC_Q = ColQwen2_5_Processor.from_pretrained(QWEN_ID)
_LIB_PREFIX = getattr(PROC_Q, "query_prefix", "") or ""
print(f"query_prefix mặc định của colpali-engine đang cài: {_LIB_PREFIX!r}")

def make_prefixed_processor():
    """Processor mới mà process_queries() tự thêm 'Query: ' (nếu thư viện chưa thêm) -> khớp giao thức train của v0.2."""
    p = ColQwen2_5_Processor.from_pretrained(QWEN_ID)
    if "query" not in (getattr(p, "query_prefix", "") or "").lower():
        p.query_prefix = QUERY_PREFIX
    return p

PROC_QP = make_prefixed_processor()
_demo = PROC_QP.process_queries(["xin chào"])
_demo_txt = PROC_QP.tokenizer.decode(_demo["input_ids"][0])
print("Ví dụ chuỗi query thực sự đưa vào model:", repr(_demo_txt))
assert _demo_txt.count("Query") == 1, "Tiền tố 'Query:' bị thiếu hoặc bị lặp!"

def query_token_info(proc, texts):
    """Trả về (ids, content_mask) cho từng query, KHỚP 1-1 với vector embedding (cùng tokenizer/process_texts).
    content_mask = True cho token thuộc phần câu hỏi (không phải tiền tố 'Query:' và không phải 10 token augmentation)."""
    prefix = getattr(proc, "query_prefix", "") or ""
    suffix = proc.query_augmentation_token * 10
    ids_all, cm_all = [], []
    for t in texts:
        full = prefix + t + suffix
        enc = proc.tokenizer(full, return_offsets_mapping=True)
        lo, hi = len(prefix), len(prefix) + len(t)
        cm = [bool(e > lo and s < hi and e > s) for (s, e) in enc["offset_mapping"]]
        ids_all.append(list(enc["input_ids"]))
        cm_all.append(np.asarray(cm, dtype=bool))
    return ids_all, cm_all

def _check_align(embs, ids, name):
    bad = sum(int(len(e) != len(i)) for e, i in zip(embs, ids))
    if bad:
        print(f"  ⚠ {name}: {bad}/{len(embs)} query lệch độ dài token/embedding -> dùng content mask dự phòng")
    return bad

def _fallback_cm(e):
    cm = np.ones(len(e), dtype=bool)
    cm[-10:] = False
    cm[:2] = False
    return cm

def build_query_pack(model, proc, texts, bs=32, name="q"):
    Q = encode(model, proc, texts, False, bs=bs)
    ids, cms = query_token_info(proc, texts)
    if _check_align(Q, ids, name):
        cms = [cm if len(cm) == len(e) else _fallback_cm(e) for e, cm in zip(Q, cms)]
    return Q, ids, cms

_QWEN_MODEL = None
def get_qwen_model():
    """Nạp ColQwen2.5 zero-shot đúng 1 lần (chỉ khi thiếu cache)."""
    global _QWEN_MODEL
    if _QWEN_MODEL is None:
        _QWEN_MODEL, _ = load_model_safe(
            lambda dt: ColQwen2_5.from_pretrained(QWEN_ID, torch_dtype=dt, device_map=DEVICE),
            PROC_Q, TEST_DOCS[:2], TEST_Q[:2], cast_mask=False,
        )
    return _QWEN_MODEL

def release_qwen_model():
    global _QWEN_MODEL
    if _QWEN_MODEL is not None:
        del _QWEN_MODEL
        _QWEN_MODEL = None
        free()

# ---- 1. Cache Giai đoạn 2 (page + query KHÔNG tiền tố). Nếu thiếu thì encode y hệt Giai đoạn 2 ----
if not os.path.exists(QWEN_TEST_CACHE):
    print(">>> Thiếu cache test Giai đoạn 2 -> encode lại (giống hệt Giai đoạn 2)")
    m = get_qwen_model()
    t0 = time.perf_counter()
    _Dt = encode(m, PROC_Q, TEST_DOCS, True, bs=QWEN_IMG_BS)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    _enc_s = (time.perf_counter() - t0) / len(TEST_DOCS)
    torch.save({
        "D_test": _Dt, "Q_test": encode(m, PROC_Q, TEST_Q, False, bs=16),
        "D_en": encode(m, PROC_Q, EN_DOCS, True, bs=QWEN_IMG_BS), "Q_en": encode(m, PROC_Q, EN_Q, False, bs=16),
        "encode_s_per_page": _enc_s, "embed_dim": int(_Dt[0].shape[-1]), "dtype": "torch.bfloat16",
    }, QWEN_TEST_CACHE)
    del _Dt
if not os.path.exists(QWEN_TRVAL_CACHE):
    print(">>> Thiếu cache train/val Giai đoạn 2 -> encode lại (giống hệt Giai đoạn 2)")
    m = get_qwen_model()
    torch.save({
        "D_tr": encode(m, PROC_Q, [r["path"] for r in sg_pages], True, bs=QWEN_IMG_BS),
        "Q_tr": encode(m, PROC_Q, tr_q, False, bs=32),
        "D_val": encode(m, PROC_Q, [r["path"] for r in val_pages], True, bs=QWEN_IMG_BS),
        "Q_val": encode(m, PROC_Q, [r["qs"][0] for r in val_pages], False, bs=32),
    }, QWEN_TRVAL_CACHE)

pack_test = torch.load(QWEN_TEST_CACHE, map_location="cpu")
D_test, Q_test_np = pack_test["D_test"], pack_test["Q_test"]
D_en, Q_en_np = pack_test.get("D_en"), pack_test.get("Q_en")
ENC_S_PER_PAGE = pack_test.get("encode_s_per_page")
pack_trval = torch.load(QWEN_TRVAL_CACHE, map_location="cpu")
D_tr, Q_tr_np = pack_trval["D_tr"], pack_trval["Q_tr"]
D_val, Q_val_np = pack_trval["D_val"], pack_trval["Q_val"]
EMB_DIM = int(D_test[0].shape[-1])

# ---- 2. Query có tiền tố "Query: " (chỉ encode query, embedding trang giữ nguyên) ----
if not os.path.exists(QPREFIX_CACHE):
    print(">>> Encode lại toàn bộ query với tiền tố 'Query: ' ...")
    m = get_qwen_model()
    _qp = {}
    for key, texts in [("test", TEST_Q), ("val", [r["qs"][0] for r in val_pages]), ("tr", tr_q), ("en", EN_Q)]:
        Q, ids, cms = build_query_pack(m, PROC_QP, texts, bs=32, name=key)
        _qp[f"Q_{key}"], _qp[f"ids_{key}"], _qp[f"cm_{key}"] = Q, ids, cms
    torch.save(_qp, QPREFIX_CACHE)
qp = torch.load(QPREFIX_CACHE, map_location="cpu", weights_only=False)
Q_test, Q_val, Q_tr, Q_en = qp["Q_test"], qp["Q_val"], qp["Q_tr"], qp["Q_en"]
C_test, C_val, C_tr, C_en = qp["cm_test"], qp["cm_val"], qp["cm_tr"], qp["cm_en"]
IDS_test = qp["ids_test"]
assert len(Q_test) == len(TEST_Q) and len(Q_tr) == len(tr_q) and len(Q_en) == len(EN_Q)

# ---- 3. Bộ test ngoài miền ViInfographicVQA (single-image test) ----
def load_ood_viinfographic():
    ood_dir = f"{DATA_DIR}/viinfographic"
    js = hf_hub_download(OOD_REPO, "data/single_test.json", repo_type="dataset", local_dir=ood_dir)
    rows = json.load(open(js, encoding="utf-8"))
    rows = [r for r in rows if isinstance(r.get("question"), str) and len(r["question"].strip()) > 10 and r.get("image_path")]
    paths = sorted({r["image_path"] for r in rows})
    local = {}
    for ip in tqdm(paths, desc="tải ảnh ViInfographicVQA"):
        local[ip] = hf_hub_download(OOD_REPO, ip, repo_type="dataset", local_dir=ood_dir)
    pos = {ip: i for i, ip in enumerate(paths)}
    random.Random(SEED).shuffle(rows)
    rows = rows[:OOD_MAX_Q]
    return [local[p] for p in paths], [r["question"].strip() for r in rows], [pos[r["image_path"]] for r in rows]

D_ood = Q_ood = C_ood = None
OOD_DOCS, OOD_Q, OOD_T = [], [], []
if RUN_OOD:
    try:
        OOD_DOCS, OOD_Q, OOD_T = load_ood_viinfographic()
        ood_h = {img_hash(p) for p in OOD_DOCS}
        print(f"OOD ViInfographicVQA: {len(OOD_DOCS)} ảnh / {len(OOD_Q)} câu hỏi | "
              f"trùng hash với V-Doc train/val/test: {len(ood_h & (vdoc_tr_h | val_h | test_h))}")
        if not os.path.exists(OOD_CACHE):
            m = get_qwen_model()
            D_o = encode(m, PROC_Q, OOD_DOCS, True, bs=QWEN_IMG_BS)
            Q_o, ids_o, cm_o = build_query_pack(m, PROC_QP, OOD_Q, bs=32, name="ood")
            torch.save({"D_ood": D_o, "Q_ood": Q_o, "ids_ood": ids_o, "cm_ood": cm_o, "T_ood": OOD_T}, OOD_CACHE)
        _po = torch.load(OOD_CACHE, map_location="cpu", weights_only=False)
        D_ood, Q_ood, C_ood = _po["D_ood"], _po["Q_ood"], _po["cm_ood"]
        assert list(_po["T_ood"]) == list(OOD_T), "Thứ tự OOD không khớp cache -> xoá file OOD_CACHE rồi chạy lại"
    except Exception as exc:
        print(f"⚠ Bỏ qua bộ OOD do lỗi: {exc}")
        traceback.print_exc()
        D_ood = Q_ood = C_ood = None

release_qwen_model()
print(f"Sẵn sàng -> D_tr {len(D_tr)} | D_val {len(D_val)} | D_test {len(D_test)} | D_en {len(D_en)} | "
      f"D_ood {len(D_ood) if D_ood else 0} | dim {EMB_DIM} | {vec_per_page(D_test):.1f} vec/trang")

In [ ]:
# =====================================================================================
# Hierarchical Token Pooling ×4 (Clavié et al., 2024 — HierarchicalTokenPooler chính thức của colpali-engine)
# Cache riêng theo từng tập; tập train/val chỉ cần cho baseline "adapter + Hier×4".
# =====================================================================================
from colpali_engine.compression.token_pooling import HierarchicalTokenPooler
HIER_POOLER = HierarchicalTokenPooler()

def hier_pool(D_list, pool_factor=4):
    out = HIER_POOLER.pool_embeddings([x.float() for x in D_list], pool_factor=pool_factor, num_workers=os.cpu_count())
    return [x.to(torch.bfloat16) for x in out]

HIER = torch.load(HIER_CACHE, map_location="cpu") if os.path.exists(HIER_CACHE) else {}
_need = {"test": D_test, "en": D_en}
if D_ood is not None:
    _need["ood"] = D_ood
if NEED_HIER_TRAIN:
    _need.update({"tr": D_tr, "val": D_val})
_changed = False
for key, D in _need.items():
    if key not in HIER or len(HIER[key]) != len(D):
        t0 = time.perf_counter()
        HIER[key] = hier_pool(D, 4)
        HIER[f"ms_{key}"] = (time.perf_counter() - t0) * 1000 / len(D)
        _changed = True
        print(f"  Hier×4 [{key}]: {len(D)} trang | {vec_per_page(HIER[key]):.1f} vec/trang | {HIER[f'ms_{key}']:.1f} ms/trang (CPU x{os.cpu_count()})")
if _changed:
    torch.save(HIER, HIER_CACHE)
print("Hier×4 sẵn sàng cho:", [k for k in HIER if not k.startswith("ms_")])

## 5. Mốc zero-shot chuẩn hoá & Hier×4
Báo cáo cả hai giao thức (có / không `'Query: '`) để thấy giao thức đổi mốc bao nhiêu. **Mọi Δ và p trong bảng cuối đều so với mốc CÓ tiền tố.**

In [ ]:
# =====================================================================================
# Mốc zero-shot chuẩn hoá: có / không tiền tố "Query: " (kiểm chứng giao thức) + Hier×4
# =====================================================================================
REF_ZS = "ZS: ColQwen2.5-3B (+'Query:')"
REF_ZS_NOPFX = "ZS: ColQwen2.5-3B (không tiền tố — giao thức v2)"
REF_HIER = "ZS: ColQwen2.5 + Hier×4 (+'Query:')"
DEVICE_NAME = torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU"

def rank_lists(Q, D, T, G=None):
    return ranks_from_scores(score_ragged_maxsim(Q, D, G_list=G), T)

def eval_triplet(Qs, Ds, G=None):
    """Qs/Ds: dict {'vi','en','ood'} -> list; trả về ranks cho từng bộ có sẵn."""
    out = {"vi": rank_lists(Qs["vi"], Ds["vi"], TEST_T)}
    if RUN_EN_CHECK and Ds.get("en") is not None and Qs.get("en") is not None:
        out["en"] = rank_lists(Qs["en"], Ds["en"], EN_T)
    if Ds.get("ood") is not None and Qs.get("ood") is not None:
        out["ood"] = rank_lists(Qs["ood"], Ds["ood"], OOD_T)
    return out

if REF_ZS_NOPFX not in RESULTS:
    r = eval_triplet({"vi": Q_test_np, "en": Q_en_np}, {"vi": D_test, "en": D_en})
    save_result(REF_ZS_NOPFX, vi=r["vi"], en=r.get("en"), vec_per_page=vec_per_page(D_test), embed_dim=EMB_DIM,
                encode_s_per_page=ENC_S_PER_PAGE, device=DEVICE_NAME, trained=False, group="zero-shot", n_seeds=1)
if REF_ZS not in RESULTS:
    r = eval_triplet({"vi": Q_test, "en": Q_en, "ood": Q_ood}, {"vi": D_test, "en": D_en, "ood": D_ood})
    save_result(REF_ZS, vi=r["vi"], en=r.get("en"), ood=r.get("ood"), vec_per_page=vec_per_page(D_test), embed_dim=EMB_DIM,
                encode_s_per_page=ENC_S_PER_PAGE, device=DEVICE_NAME, trained=False, group="zero-shot", n_seeds=1)
if REF_HIER not in RESULTS:
    r = eval_triplet({"vi": Q_test, "en": Q_en, "ood": Q_ood}, {"vi": HIER["test"], "en": HIER["en"], "ood": HIER.get("ood")})
    save_result(REF_HIER, vi=r["vi"], en=r.get("en"), ood=r.get("ood"), vec_per_page=vec_per_page(HIER["test"]), embed_dim=EMB_DIM,
                ms_per_page=HIER.get("ms_test"), device=f"CPU x{os.cpu_count()}", trained=False, group="zero-shot", n_seeds=1)

def _nd(name, key="vi"):
    v = RESULTS[name].get(key)
    return None if v is None else round(float(np.mean(seed_scores(v))), 2)

print("Kiểm chứng giao thức query (nDCG@5):")
for n in [REF_ZS_NOPFX, REF_ZS, REF_HIER]:
    print(f"  {n:55s} VI {_nd(n)} | EN {_nd(n, 'en')} | OOD {_nd(n, 'ood')}")
print("  Tham chiếu model card vidore/colqwen2.5-v0.2 trên DocVQA (ViDoRe v1): 63.22 "
      "(protocol MTEB/vidore-benchmark; bộ 500 câu của notebook có 451 câu khác nhau -> có thể lệch nhẹ)")
p_pfx = paired_randomization_p(pq_mean(RESULTS[REF_ZS]["vi"]), pq_mean(RESULTS[REF_ZS_NOPFX]["vi"]))
print(f"  Δ VI do tiền tố = {_nd(REF_ZS) - _nd(REF_ZS_NOPFX):+.2f} (randomization p = {fmt_p(p_pfx)})")

## 6. Baseline fine-tune nhẹ trên embedding (E1, E1b, E2, E3)
Mọi baseline chỉ dùng **CE** (+ L1 gate với E2) — đúng như `Q-B` của Giai đoạn 2 (KD không tạo khác biệt). Tài liệu = Hier×4 cố định (E1, E1b, E2) hoặc không nén (E3).

In [ ]:
# =====================================================================================
# Hard negatives dùng chung cho MỌI mô hình train (notebook baseline và notebook v3 dùng cùng 1 file .npy)
# =====================================================================================
# ---- Hard negatives: khai thác bằng ColQwen2.5 zero-shot (+'Query:') trên trang KHÔNG nén (dùng chung mọi baseline) ----
VDOC_TR_T = np.asarray(tr_t, dtype=np.int64)
_hid = {}
VDOC_TR_H = np.asarray([_hid.setdefault(r["h"], len(_hid)) for r in sg_pages], dtype=np.int64)
HARD_P_PATH = f"{SAVE_DIR}/stage3_hardneg_prefix_k{HARD_NEG}_tr{N_TRAIN_PAGES}x{MAX_Q_PER_PAGE}.npy"
if os.path.exists(HARD_P_PATH):
    HARD_P = np.load(HARD_P_PATH)
else:
    HARD_P = mine_hard_negatives(Q_tr, D_tr, VDOC_TR_T, VDOC_TR_H, k=HARD_NEG)
    np.save(HARD_P_PATH, HARD_P)
# =====================================================================================
# Baseline fine-tune NHẸ trên embedding (cùng dữ liệu, cùng hard negatives, cùng lịch train, 3 seed)
#   E1  Linear-Q  + Hier×4 : 1 phép chiếu tuyến tính 128×128 phía query (khởi tạo đồng nhất), ~16,5K tham số
#   E1b Linear-QD + Hier×4 : thêm 1 phép chiếu tuyến tính phía tài liệu (sau pooling), ~33K tham số
#   E2  Query-Adapter v2 + Hier×4 : đúng nhánh câu hỏi của DynamicMultiVectorAdapter (subword attn + gate), tài liệu = Hier×4
#   E3  Linear-Q (không nén, 733 vec) : mốc "thích nghi rẻ nhất" không nén
# =====================================================================================
class LinearAdapter(nn.Module):
    def __init__(self, dim=128, doc_side=False):
        super().__init__()
        self.lq = nn.Linear(dim, dim)
        nn.init.eye_(self.lq.weight); nn.init.zeros_(self.lq.bias)
        self.ld = None
        if doc_side:
            self.ld = nn.Linear(dim, dim)
            nn.init.eye_(self.ld.weight); nn.init.zeros_(self.ld.bias)
        self.doc_mode = "none"
        self.use_doc_adapter = doc_side
        self.use_subword_attn = False
        self.use_query_gate = False

    def forward_docs(self, E_d, d_mask):
        m = d_mask.bool()
        x = F.normalize(E_d.float(), p=2, dim=-1)
        if self.ld is not None:
            x = F.normalize(self.ld(x), p=2, dim=-1)
        return x * m.float().unsqueeze(-1), m

    def forward_queries(self, E_q, q_mask):
        m = q_mask.float().unsqueeze(-1)
        q0 = F.normalize(E_q.float(), p=2, dim=-1) * m
        q = F.normalize(self.lq(q0), p=2, dim=-1) * m
        return q, None, torch.zeros(1, device=E_q.device), q0

    def forward(self, E_q, q_mask, E_d, d_mask):
        d, dm = self.forward_docs(E_d, d_mask)
        q, _, gd, q0 = self.forward_queries(E_q, q_mask)
        s = score_masked_maxsim(q, q_mask, d, dm, gates=None, q_chunk_size=SCORE_CHUNK, d_chunk_size=DOC_CHUNK)
        return s, gd, q, q0

def train_generic(make_net, Q_tr_list, D_tr_list, tr_t_np, tr_h_ids, hard_neg_np, Q_val_list, D_val_list, seed):
    """Giống hệt vòng train Giai đoạn 2 (AdamW, warmup + cosine, BQ, hard negatives, mask false-negative,
    chọn epoch theo val nDCG@5) nhưng CHỈ dùng CE (+ L1 gate nếu có) để làm baseline sạch."""
    torch.manual_seed(seed)
    gen = np.random.default_rng(seed)
    net = make_net().to(DEVICE)
    Nq = len(Q_tr_list)
    steps_per_epoch = int(np.ceil(Nq / BQ))
    opt = torch.optim.AdamW(net.parameters(), lr=LR_ADAPTER, weight_decay=WEIGHT_DECAY)
    sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps=steps_per_epoch * WARMUP_EPOCHS,
                                            num_training_steps=steps_per_epoch * EPOCHS)
    best, best_state = -1.0, None
    for ep in range(EPOCHS):
        net.train()
        perm = gen.permutation(Nq)
        ep_loss, n_steps = 0.0, 0
        for i in range(0, Nq, BQ):
            idx = perm[i:i + BQ]
            pos = tr_t_np[idx]
            cand = np.unique(np.concatenate([pos, hard_neg_np[idx].reshape(-1)]))
            tgt = torch.as_tensor(np.searchsorted(cand, pos), dtype=torch.long, device=DEVICE)
            fn = torch.as_tensor((tr_h_ids[cand][None, :] == tr_h_ids[pos][:, None]) & (cand[None, :] != pos[:, None]), device=DEVICE)
            q_pad, q_mask = _pad_with_mask([Q_tr_list[j] for j in idx])
            d_pad, d_mask = _pad_with_mask([D_tr_list[j] for j in cand])
            s, g_delta, _, _ = net(q_pad.to(DEVICE), q_mask.to(DEVICE), d_pad.to(DEVICE), d_mask.to(DEVICE))
            loss = F.cross_entropy(s.masked_fill(fn, -1e4) / TEMP_CE, tgt)
            if getattr(net, "use_query_gate", False):
                loss = loss + 0.005 * g_delta.abs().mean()
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
            sched.step()
            ep_loss += float(loss.item()); n_steps += 1
        nd = float(metrics(eval_adapter_ragged(net, Q_val_list, D_val_list, VAL_T)[0])["nDCG@5"])
        if nd > best:
            best, best_state = nd, copy.deepcopy(net.state_dict())
        print(f"  epoch {ep + 1}/{EPOCHS}  loss {ep_loss / max(1, n_steps):.4f}  val nDCG@5 {nd:.1f}")
    net.load_state_dict(best_state)
    return net.eval()

def n_params(net):
    """Chỉ đếm tham số THỰC SỰ được dùng (có gradient sau bước train cuối)."""
    return int(sum(p.numel() for p in net.parameters() if p.grad is not None))

def run_embed_baseline(name, make_net, D_tr_use, D_val_use, D_test_use, D_en_use, D_ood_use, vec_src, note):
    if name in RESULTS:
        print("✓ đã có:", name)
        return
    vi, en, ood, t0 = [], [], [], time.time()
    for seed in SEEDS:
        print(f"\n>>> {name} | seed {seed}")
        net = train_generic(make_net, Q_tr, D_tr_use, VDOC_TR_T, VDOC_TR_H, HARD_P, Q_val, D_val_use, seed)
        vi.append(eval_adapter_ragged(net, Q_test, D_test_use, TEST_T)[0])
        if RUN_EN_CHECK and D_en_use is not None:
            en.append(eval_adapter_ragged(net, Q_en, D_en_use, EN_T)[0])
        if D_ood_use is not None and Q_ood is not None:
            ood.append(eval_adapter_ragged(net, Q_ood, D_ood_use, OOD_T)[0])
        n_par = n_params(net)
        del net
        free()
    save_result(name, vi=vi, en=en or None, ood=ood or None, vec_per_page=vec_per_page(vec_src), embed_dim=EMB_DIM,
                device=DEVICE_NAME, trained=True, group="fine-tune nhẹ (embedding)", n_seeds=len(SEEDS),
                n_params=n_par, note=note, train_min_per_seed=(time.time() - t0) / 60 / len(SEEDS))
    print(name, "| VI nDCG@5 theo seed:", [round(x, 1) for x in seed_scores(vi)])

NAME_E1 = "FT-E1: Linear-Q + Hier×4 (V-Doc, CE)"
NAME_E1B = "FT-E1b: Linear-Q + Linear-D + Hier×4 (V-Doc, CE)"
NAME_E2 = "FT-E2: Query-Adapter v2 (subword+gate) + Hier×4 (V-Doc, CE)"
NAME_E3 = "FT-E3: Linear-Q không nén 733 vec (V-Doc, CE)"
if RUN_EMBED_BASELINES:
    H = HIER
    run_embed_baseline(NAME_E1, lambda: LinearAdapter(EMB_DIM, doc_side=False), H["tr"], H["val"], H["test"], H["en"], H.get("ood"),
                       H["test"], "query-side linear; docs = Hier×4 cố định")
    run_embed_baseline(NAME_E1B, lambda: LinearAdapter(EMB_DIM, doc_side=True), H["tr"], H["val"], H["test"], H["en"], H.get("ood"),
                       H["test"], "linear 2 phía; docs = Hier×4 rồi chiếu")
    run_embed_baseline(NAME_E2, lambda: DynamicMultiVectorAdapter(dim=EMB_DIM, rank=32, window_size=8, dropout=0.1, doc_mode="none",
                                                                  use_doc_adapter=False, use_subword_attn=True, use_query_gate=True),
                       H["tr"], H["val"], H["test"], H["en"], H.get("ood"), H["test"], "nhánh query của v2; docs = Hier×4 cố định")
    run_embed_baseline(NAME_E3, lambda: LinearAdapter(EMB_DIM, doc_side=False), D_tr, D_val, D_test, D_en, D_ood,
                       D_test, "query-side linear; docs đầy đủ 733 vec")

## 7. Baseline fine-tune mạnh: LoRA r=32 trên ColQwen2.5-3B
- Tiếp tục train adapter LoRA của `colqwen2.5-v0.2` (vision tower đóng băng), loss gốc ColPali `softplus(max_neg − pos)` với in-batch + 1 hard negative/câu, mask false-negative theo hash.
- Master weight FP32 + autocast bf16, gradient checkpointing, tự giảm micro-batch khi OOM, giới hạn `LORA_MAX_HOURS`, lưu checkpoint để chạy tiếp.

In [ ]:
# =====================================================================================
# Baseline fine-tune MẠNH: LoRA r=32 trên chính ColQwen2.5-3B, cùng 2.000 trang / 4.000 câu V-Doc
#   - Tiếp tục huấn luyện adapter LoRA của vidore/colqwen2.5-v0.2 (r=32, α=32, LM + custom_text_proj; vision đóng băng)
#     nếu adapter được nạp dưới dạng PEFT; nếu không có tham số lora_* thì tạo LoRA mới cùng cấu hình.
#   - Loss gốc của ColPali: softplus(max_neg − pos) với in-batch negatives + 1 hard negative/câu, mask false-negative theo hash.
#   - 1 seed (chi phí ~1–2 giờ trên L4); có giới hạn thời gian LORA_MAX_HOURS, tự giảm micro-batch nếu OOM, lưu checkpoint.
# =====================================================================================
from transformers import get_linear_schedule_with_warmup

LORA_NAME_FULL = f"FT-LoRA: ColQwen2.5 LoRA r=32 (V-Doc, seed {LORA_SEED}) — 733 vec"
LORA_NAME_HIER = f"FT-LoRA: ColQwen2.5 LoRA r=32 (V-Doc, seed {LORA_SEED}) + Hier×4"
LORA_EMB_CACHE = f"{SAVE_DIR}/stage3_lora_colqwen25_vdoc_seed{LORA_SEED}_emb.pt"
LORA_CKPT = f"{SAVE_DIR}/stage3_lora_colqwen25_vdoc_seed{LORA_SEED}_trainable.pt"

def lora_prepare(model):
    for p in model.parameters():
        p.requires_grad_(False)
    named = [(n, p) for n, p in model.named_parameters() if "lora_" in n]
    mode = "tiếp tục adapter LoRA của colqwen2.5-v0.2"
    if not named:
        from peft import LoraConfig, get_peft_model
        cfg = LoraConfig(r=32, lora_alpha=32, lora_dropout=0.1, init_lora_weights="gaussian",
                         target_modules=r".*language_model.*\.(down_proj|gate_proj|up_proj|k_proj|q_proj|v_proj|o_proj)$")
        model = get_peft_model(model, cfg)
        named = [(n, p) for n, p in model.named_parameters() if "lora_" in n]
        mode = "LoRA mới r=32 (không tìm thấy tham số lora_* trong checkpoint)"
    for _, p in named:
        p.requires_grad_(True)
    for n, p in model.named_parameters():
        if "custom_text_proj" in n:
            p.requires_grad_(True)
    trainable = [p for p in model.parameters() if p.requires_grad]
    for p in trainable:
        p.data = p.data.float()  # master weights FP32, tính toán bf16 qua autocast
    return model, trainable, mode

def to_dev_bf16(batch):
    """Đưa batch lên GPU; mọi tensor số thực -> bf16 (không phụ thuộc dtype của tham số đầu tiên khi đang có master FP32)."""
    out = {}
    for k, v in batch.items():
        if k == "labels":
            continue
        if torch.is_tensor(v):
            out[k] = v.to(DEVICE, dtype=torch.bfloat16) if v.is_floating_point() else v.to(DEVICE)
        else:
            out[k] = v
    return out

def lora_batch_loss(model, q_idx, rng):
    pos = VDOC_TR_T[q_idx]
    k = min(3, HARD_P.shape[1])
    hn = np.asarray([HARD_P[q][rng.integers(0, k)] for q in q_idx], dtype=np.int64)
    cand = np.unique(np.concatenate([pos, hn]))
    tgt = torch.as_tensor(np.searchsorted(cand, pos), device=DEVICE)
    bd = to_dev_bf16(PROC_QP.process_images([load_img(sg_pages[c]["path"]) for c in cand]))
    bq = to_dev_bf16(PROC_QP.process_queries([tr_q[j] for j in q_idx]))
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
        d = model(**bd)
        q = model(**bq)
    d, q = d.float(), q.float()
    dm = bd["attention_mask"].bool()
    sim = torch.einsum("aid,cjd->acij", q, d).masked_fill(~dm[None, :, None, :], -1e4)
    s = sim.amax(dim=-1).sum(dim=-1)  # (Nq, Nd); token query padding = vector 0 -> đóng góp 0
    fn = torch.as_tensor((VDOC_TR_H[cand][None, :] == VDOC_TR_H[pos][:, None]), device=DEVICE)
    pos_s = s[torch.arange(len(q_idx), device=DEVICE), tgt]
    neg = s.masked_fill(fn, -1e4).amax(dim=1)  # fn đã gồm chính trang đúng (cùng hash)
    return F.softplus(neg - pos_s).mean()

def train_lora(model, trainable):
    rng = np.random.default_rng(LORA_SEED)
    torch.manual_seed(LORA_SEED)
    n_q = len(tr_q)
    eff = LORA_EFFECTIVE_BS
    steps_total = int(np.ceil(n_q / eff)) * LORA_EPOCHS
    opt = torch.optim.AdamW(trainable, lr=LORA_LR, weight_decay=0.0)
    sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=max(1, int(0.05 * steps_total)), num_training_steps=steps_total)
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except Exception as exc:
        print("  (không bật được gradient checkpointing:", exc, ")")
    model.train()
    micro = LORA_MICRO_BS
    t0, step, log_loss = time.time(), 0, []
    stopped_early = False
    for ep in range(LORA_EPOCHS):
        order = rng.permutation(n_q)
        g = 0
        pbar = tqdm(total=int(np.ceil(n_q / eff)), desc=f"LoRA epoch {ep + 1}/{LORA_EPOCHS}")
        while g < n_q:
            group = order[g:g + eff]
            try:
                opt.zero_grad(set_to_none=True)
                gl = 0.0
                n_micro = int(np.ceil(len(group) / micro))
                for mi in range(n_micro):
                    loss = lora_batch_loss(model, group[mi * micro:(mi + 1) * micro], rng) / n_micro
                    loss.backward()
                    gl += float(loss.item())
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                opt.step()
                sched.step()
            except torch.OutOfMemoryError:
                opt.zero_grad(set_to_none=True)
                free()
                if micro == 1:
                    raise
                micro = max(1, micro // 2)
                print(f"  ⚠ OOM -> giảm micro-batch xuống {micro} và chạy lại nhóm này")
                continue
            g += eff
            step += 1
            log_loss.append(gl)
            pbar.update(1)
            pbar.set_postfix(loss=f"{np.mean(log_loss[-20:]):.4f}", micro=micro)
            if (time.time() - t0) / 3600 > LORA_MAX_HOURS:
                print(f"  ⏱ Dừng sớm sau {step} bước vì vượt LORA_MAX_HOURS={LORA_MAX_HOURS}")
                stopped_early = True
                break
        pbar.close()
        if stopped_early:
            break
    model.eval()
    try:
        model.gradient_checkpointing_disable()
    except Exception:
        pass
    for p in trainable:
        p.data = p.data.to(torch.bfloat16)
    del opt
    free()
    return {"steps": step, "minutes": (time.time() - t0) / 60, "final_loss": float(np.mean(log_loss[-20:])) if log_loss else None,
            "stopped_early": stopped_early, "micro_bs": micro}

if RUN_LORA and torch.cuda.is_available() and not (LORA_NAME_FULL in RESULTS and LORA_NAME_HIER in RESULTS):
    if not os.path.exists(LORA_EMB_CACHE):
        print(">>> Nạp ColQwen2.5-3B để fine-tune LoRA ...")
        model_l = ColQwen2_5.from_pretrained(QWEN_ID, torch_dtype=torch.bfloat16, device_map=DEVICE)
        model_l, trainable, lora_mode = lora_prepare(model_l)
        n_tr = sum(p.numel() for p in trainable)
        print(f"  Chế độ: {lora_mode} | tham số train: {n_tr:,}")
        if os.path.exists(LORA_CKPT):
            ck = torch.load(LORA_CKPT, map_location="cpu")
            missing = model_l.load_state_dict(ck["state"], strict=False)
            for p in trainable:
                p.data = p.data.to(torch.bfloat16)
            model_l.eval()
            info = ck["info"]
            print("  Nạp checkpoint LoRA đã train:", info)
        else:
            info = train_lora(model_l, trainable)
            info.update({"mode": lora_mode, "n_trainable": int(n_tr)})
            state = {n: p.detach().cpu() for n, p in model_l.named_parameters() if p.requires_grad}
            torch.save({"state": state, "info": info}, LORA_CKPT)
            print("  Đã lưu checkpoint LoRA:", LORA_CKPT, "|", info)
        pack_l = {"info": info}
        t0 = time.perf_counter()
        pack_l["D_test"] = encode(model_l, PROC_Q, TEST_DOCS, True, bs=QWEN_IMG_BS)
        pack_l["encode_s_per_page"] = (time.perf_counter() - t0) / len(TEST_DOCS)
        pack_l["Q_test"] = encode(model_l, PROC_QP, TEST_Q, False, bs=32)
        pack_l["D_val"] = encode(model_l, PROC_Q, [r["path"] for r in val_pages], True, bs=QWEN_IMG_BS)
        pack_l["Q_val"] = encode(model_l, PROC_QP, [r["qs"][0] for r in val_pages], False, bs=32)
        if RUN_EN_CHECK:
            pack_l["D_en"] = encode(model_l, PROC_Q, EN_DOCS, True, bs=QWEN_IMG_BS)
            pack_l["Q_en"] = encode(model_l, PROC_QP, EN_Q, False, bs=32)
        if D_ood is not None:
            pack_l["D_ood"] = encode(model_l, PROC_Q, OOD_DOCS, True, bs=QWEN_IMG_BS)
            pack_l["Q_ood"] = encode(model_l, PROC_QP, OOD_Q, False, bs=32)
        torch.save(pack_l, LORA_EMB_CACHE)
        del model_l, trainable
        free()
    pack_l = torch.load(LORA_EMB_CACHE, map_location="cpu", weights_only=False)
    info = pack_l["info"]
    val_nd = float(metrics(rank_lists(pack_l["Q_val"], pack_l["D_val"], VAL_T))["nDCG@5"])
    print(f"  LoRA val nDCG@5 = {val_nd:.1f} | info = {info}")
    Qs = {"vi": pack_l["Q_test"], "en": pack_l.get("Q_en"), "ood": pack_l.get("Q_ood")}
    if LORA_NAME_FULL not in RESULTS:
        r = eval_triplet(Qs, {"vi": pack_l["D_test"], "en": pack_l.get("D_en"), "ood": pack_l.get("D_ood")})
        save_result(LORA_NAME_FULL, vi=[r["vi"]], en=[r["en"]] if "en" in r else None, ood=[r["ood"]] if "ood" in r else None,
                    vec_per_page=vec_per_page(pack_l["D_test"]), embed_dim=EMB_DIM, encode_s_per_page=pack_l["encode_s_per_page"],
                    device=DEVICE_NAME, trained=True, group="fine-tune backbone (LoRA)", n_seeds=1,
                    n_params=info.get("n_trainable"), note=str(info))
    if LORA_NAME_HIER not in RESULTS:
        t0 = time.perf_counter()
        Dh = {"vi": hier_pool(pack_l["D_test"])}
        ms_h = (time.perf_counter() - t0) * 1000 / len(pack_l["D_test"])
        Dh["en"] = hier_pool(pack_l["D_en"]) if pack_l.get("D_en") is not None else None
        Dh["ood"] = hier_pool(pack_l["D_ood"]) if pack_l.get("D_ood") is not None else None
        r = eval_triplet(Qs, Dh)
        save_result(LORA_NAME_HIER, vi=[r["vi"]], en=[r["en"]] if "en" in r else None, ood=[r["ood"]] if "ood" in r else None,
                    vec_per_page=vec_per_page(Dh["vi"]), embed_dim=EMB_DIM, ms_per_page=ms_h, device=f"CPU x{os.cpu_count()}",
                    trained=True, group="fine-tune backbone (LoRA)", n_seeds=1, n_params=info.get("n_trainable"), note=str(info))
    for n in [LORA_NAME_FULL, LORA_NAME_HIER]:
        print(f"  {n}: VI {np.mean(seed_scores(RESULTS[n]['vi'])):.2f}")
elif RUN_LORA and not torch.cuda.is_available():
    print("⚠ Bỏ qua LoRA: cần GPU.")

## 8. Bảng kết quả baseline (Δ, randomization p, Holm, EN, OOD)

In [ ]:
# =====================================================================================
# Bảng tổng hợp chuẩn hoá: Δ so với zero-shot (+'Query:') và Hier×4, randomization p + Holm (toàn bộ họ so sánh),
# CI95 bootstrap, EN DocVQA, OOD ViInfographicVQA, nửa test "xa train nhất" (nếu có)
# =====================================================================================
V2_IMPORT = {
    "SOTA: ColQwen2.5-3B (zero-shot)": ("[v2] ZS: ColQwen2.5-3B (không tiền tố)", False),
    "ColQwen2.5 + Hierarchical pooling ×4 (không train, CPU)": ("[v2] ZS: ColQwen2.5 + Hier×4 (không tiền tố)", False),
    "Q-A: ColQwen2.5 + DynMerge ×4 (chưa train, GPU)": ("[v2] Q-A: DynMerge ×4 chưa train", False),
    "Q-B: ColQwen2.5 + DynMerge ×4 + Adapter (Pure CE, V-Doc)": ("[v2] Q-B: DynMerge ×4 + Adapter (CE)", True),
    "Q-C: ColQwen2.5 + DynMerge ×4 + Adapter (+ KD & Anchor, V-Doc)": ("[v2] Q-C: DynMerge ×4 + Adapter (+KD)", True),
    "Q-Full: ColQwen2.5 (733 vec) + Dual Adapter (+ KD & Anchor, V-Doc)": ("[v2] Q-Full: Dual Adapter 733 vec", True),
    "Q-C-Mix: DynMerge ×4 + Adapter (+ KD, V-Doc + 5CD-AI)": ("[v2] Q-C-Mix (5CD cột image)", True),
    "Q-Full-Mix: ColQwen2.5 (733 vec) + Dual Adapter (+ KD, V-Doc + 5CD-AI)": ("[v2] Q-Full-Mix (5CD cột image)", True),
}

def import_v2_rows():
    out = {}
    for old, (new, trained) in V2_IMPORT.items():
        if old in RESULTS_V2:
            v = dict(RESULTS_V2[old])
            v.update(trained=trained, group="v2 (không tiền tố — tham chiếu)")
            out[new] = v
    return out

def _time_str(v):
    if v.get("encode_s_per_page") is not None:
        return f"{v['encode_s_per_page']:.2f} s/trang encode ({v.get('device', '')})"
    if v.get("ms_per_page") is not None:
        return f"{v['ms_per_page']:.2f} ms/trang ({v.get('device', '')})"
    return None

def build_master_table(R, order, ref_zs=REF_ZS, ref_hier=REF_HIER, far_mask=None):
    rows, p_cells = [], []
    ref_vi = {k: pq_mean(R[k]["vi"]) for k in (ref_zs, ref_hier) if k in R}
    ref_en = pq_mean(R[ref_zs]["en"]) if ref_zs in R and R[ref_zs].get("en") else None
    ref_ood = pq_mean(R[ref_zs]["ood"]) if ref_zs in R and R[ref_zs].get("ood") else None
    for name in [n for n in order if n in R]:
        v = R[name]
        vi = pq_mean(v["vi"])
        sc = seed_scores(v["vi"])
        vpp = v.get("vec_per_page")
        dim = int(v.get("embed_dim") or 128)
        lo, hi = boot_ci(vi)
        row = {
            "Mô hình": name, "Nhóm": v.get("group"), "Train V-Doc": "✓" if v.get("trained") else "✗", "Seeds": len(sc),
            "Tham số train": v.get("n_params"),
            "Vec/trang": round(float(vpp), 1) if vpp else None,
            "MB/1000 trang": round(MB_PER_1K(float(vpp), dim), 1) if vpp else None,
            "VI nDCG@5": f"{np.mean(sc):.1f}" + (f" ± {np.std(sc, ddof=1):.1f}" if len(sc) > 1 else ""),
            "VI R@1": round(float(np.mean(seed_scores(v["vi"], "R@1"))), 1),
            "VI R@5": round(float(np.mean(seed_scores(v["vi"], "R@5"))), 1),
            "CI95": f"[{lo:.1f}, {hi:.1f}]",
        }
        for ref, col in [(ref_zs, "ZS"), (ref_hier, "Hier×4")]:
            if ref in ref_vi and name != ref and len(ref_vi[ref]) == len(vi):
                row[f"Δ vs {col}"] = round(float((vi - ref_vi[ref]).mean()), 1)
                row[f"p vs {col}"] = paired_randomization_p(vi, ref_vi[ref])
                p_cells.append((len(rows), f"p vs {col}"))
        if v.get("en"):
            en = pq_mean(v["en"])
            row["EN nDCG@5"] = round(float(en.mean()), 1)
            if ref_en is not None and name != ref_zs and len(ref_en) == len(en):
                row["Δ EN"] = round(float((en - ref_en).mean()), 1)
                row["p EN"] = paired_randomization_p(en, ref_en)
                p_cells.append((len(rows), "p EN"))
        if v.get("ood"):
            od = pq_mean(v["ood"])
            row["OOD nDCG@5"] = round(float(od.mean()), 1)
            if ref_ood is not None and name != ref_zs and len(ref_ood) == len(od):
                row["Δ OOD"] = round(float((od - ref_ood).mean()), 1)
                row["p OOD"] = paired_randomization_p(od, ref_ood)
                p_cells.append((len(rows), "p OOD"))
        if far_mask is not None and ref_zs in ref_vi:
            row["VI xa-train nDCG@5"] = round(float(vi[far_mask].mean()), 1)
            if name != ref_zs:
                row["Δ xa-train"] = round(float((vi[far_mask] - ref_vi[ref_zs][far_mask]).mean()), 1)
        row["Thời gian"] = _time_str(v)
        rows.append(row)
    adj = holm_adjust([rows[i][c] for i, c in p_cells])
    for (i, c), a in zip(p_cells, adj):
        rows[i][c + " (Holm)"] = fmt_p(a)
        rows[i][c] = fmt_p(rows[i][c])
    df = pd.DataFrame(rows)
    lead = [c for c in ["Mô hình", "Nhóm", "Train V-Doc", "Seeds", "Tham số train", "Vec/trang", "MB/1000 trang",
                        "VI nDCG@5", "VI R@1", "VI R@5", "CI95", "Δ vs ZS", "p vs ZS", "p vs ZS (Holm)",
                        "Δ vs Hier×4", "p vs Hier×4", "p vs Hier×4 (Holm)", "EN nDCG@5", "Δ EN", "p EN", "p EN (Holm)",
                        "OOD nDCG@5", "Δ OOD", "p OOD", "p OOD (Holm)", "VI xa-train nDCG@5", "Δ xa-train", "Thời gian"] if c in df.columns]
    return df[lead]
# =====================================================================================
# Bảng kết quả notebook baseline (+ tham chiếu v2). Bảng tổng hợp CUỐI CÙNG nằm ở notebook stage3_adapter_v3_fixed.
# =====================================================================================
R_ALL = {**import_v2_rows(), **RESULTS}
ORDER_A = [REF_ZS_NOPFX, REF_ZS, REF_HIER,
           "[v2] ZS: ColQwen2.5-3B (không tiền tố)", "[v2] ZS: ColQwen2.5 + Hier×4 (không tiền tố)",
           "[v2] Q-C: DynMerge ×4 + Adapter (+KD)", "[v2] Q-Full: Dual Adapter 733 vec",
           NAME_E1, NAME_E1B, NAME_E2, NAME_E3, LORA_NAME_HIER, LORA_NAME_FULL]
df_a = build_master_table(R_ALL, ORDER_A)
df_a.to_csv(f"{SAVE_DIR}/results_stage3_baselines_table.csv", index=False)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
display(df_a)
print("\nĐọc bảng:")
print(" - So sánh CÔNG BẰNG chỉ trong cùng cột 'Train V-Doc'; Δ/p tính với ZS (+'Query:') và Hier×4 (+'Query:').")
print(" - Dòng [v2] dùng query KHÔNG tiền tố (giao thức cũ) — chỉ để đối chiếu, không dùng làm mốc.")
print(" - p = randomization test có cặp (B = %d); cột '(Holm)' đã hiệu chỉnh cho toàn bộ họ so sánh trong bảng." % N_PERM)
print(" - LoRA chạy 1 seed (chi phí GPU); các baseline embedding chạy %d seed." % len(SEEDS))